<a href="https://colab.research.google.com/github/eman-kanwal61/offline-letter-recognition/blob/main/Character_Recognition_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖊️ Offline Character Recognition System

Train and test a handwritten character recognition model (digits 0-9 + letters A-Z).

**Features:**
- Uses EMNIST dataset with proper image orientation handling
- Deep neural network (512→256→128) for high accuracy
- Interactive drawing canvas for testing

---

## 1️⃣ Install Dependencies

In [ ]:
!pip install torch torchvision scikit-learn pillow numpy matplotlib -q
print("✅ Dependencies installed!")

✅ Dependencies installed!


## 2️⃣ Define Model & Preprocessing

In [ ]:
import pickle
import os
import numpy as np
from PIL import Image, ImageOps
from sklearn.neural_network import MLPClassifier

class CharacterRecognitionModel:
    def __init__(self, model_path="model.pkl"):
        self.model_path = model_path
        self.model = None
        self.mapping = {}

    def train_new(self, X, y):
        """Train a deep MLPClassifier for high accuracy."""
        print("   Architecture: 784 → 512 → 256 → 128 → output")
        self.model = MLPClassifier(
            hidden_layer_sizes=(512, 256, 128),  # Deep network
            max_iter=100,
            alpha=0.0001,
            solver='adam',
            verbose=True,
            random_state=42,
            learning_rate_init=0.001,
            learning_rate='adaptive',
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            batch_size=256
        )
        self.model.fit(X, y)
        self.save_model()

    def save_model(self):
        with open(self.model_path, 'wb') as f:
            pickle.dump({'model': self.model, 'mapping': self.mapping}, f)
        print(f"✅ Model saved to {self.model_path}")

    def load_model(self):
        if os.path.exists(self.model_path):
            with open(self.model_path, 'rb') as f:
                data = pickle.load(f)
                self.model = data['model']
                self.mapping = data.get('mapping', {})
            print("✅ Model loaded!")
            return True
        return False

    def predict(self, X):
        if not self.model:
            return None, 0.0
        pred_idx = self.model.predict(X)[0]
        if self.mapping and pred_idx in self.mapping:
            return self.mapping[pred_idx], 1.0
        return str(pred_idx), 1.0

def preprocess_image_for_emnist(image_pil):
    """
    Preprocess image to match EMNIST training format.
    CRITICAL: EMNIST images are transposed, so we transpose our input too!
    """
    # Convert to grayscale
    if image_pil.mode != 'L':
        image_pil = image_pil.convert('L')

    # Invert if white background
    img_array = np.array(image_pil)
    if np.mean(img_array) > 127:
        image_pil = ImageOps.invert(image_pil)

    # Find bounding box and crop
    bbox = image_pil.getbbox()
    if not bbox:
        return np.zeros((1, 784))

    cropped = image_pil.crop(bbox)

    # Resize to fit in 20x20 with 4px margin (standard MNIST/EMNIST format)
    w, h = cropped.size
    ratio = min(20 / w, 20 / h)
    new_w, new_h = int(w * ratio), int(h * ratio)
    resized = cropped.resize((new_w, new_h), Image.Resampling.LANCZOS)

    # Center in 28x28 image
    new_img = Image.new("L", (28, 28), "black")
    paste_x = (28 - new_w) // 2
    paste_y = (28 - new_h) // 2
    new_img.paste(resized, (paste_x, paste_y))

    # CRITICAL: Transpose to match EMNIST format!
    # EMNIST stores images transposed, so we need to match that
    new_img = new_img.transpose(Image.Transpose.TRANSPOSE)

    # Normalize and flatten
    img_array = np.array(new_img).astype('float32') / 255.0
    return img_array.reshape(1, -1)

print("✅ Model and preprocessing defined!")

✅ Model and preprocessing defined!


## 3️⃣ Train the Model

Downloads EMNIST and trains a deep neural network. ⏱️ Takes 3-8 minutes.

In [ ]:
from torchvision.datasets import EMNIST
from sklearn.model_selection import train_test_split
import time

def get_emnist_mapping():
    """EMNIST ByMerge: 0-9, A-Z, and distinct lowercase letters"""
    mapping = {}
    for i in range(10):
        mapping[i] = str(i)
    for i in range(26):
        mapping[i + 10] = chr(65 + i)  # A-Z
    extra_lower = ['a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't']
    for i, char in enumerate(extra_lower):
        mapping[i + 36] = char
    return mapping

def train():
    print("🚀 Starting training...")
    start_time = time.time()

    print("\n📦 Downloading EMNIST dataset...")
    train_dataset = EMNIST(root='./data', split='bymerge', train=True, download=True)
    test_dataset = EMNIST(root='./data', split='bymerge', train=False, download=True)

    X_train = train_dataset.data.numpy()
    y_train = train_dataset.targets.numpy()
    X_test = test_dataset.data.numpy()
    y_test = test_dataset.targets.numpy()

    print(f"✅ Loaded {len(X_train)} training and {len(X_test)} test samples")
    print(f"   Classes: {len(np.unique(y_train))}")

    # NOTE: We do NOT transpose training data here!
    # EMNIST is stored transposed - we keep it that way and transpose our inputs instead

    # Flatten and normalize
    X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
    X_test = X_test.reshape(-1, 784).astype('float32') / 255.0

    print("\n🧠 Training deep neural network...")
    model_wrapper = CharacterRecognitionModel()
    model_wrapper.mapping = get_emnist_mapping()
    model_wrapper.train_new(X_train, y_train)

    # Evaluate
    test_acc = model_wrapper.model.score(X_test, y_test)
    elapsed = time.time() - start_time

    print(f"\n" + "="*50)
    print(f"🎯 Test Accuracy: {test_acc * 100:.2f}%")
    print(f"⏱️ Training time: {elapsed:.1f} seconds")
    print("="*50)

    return model_wrapper

# Train!
trained_model = train()

🚀 Starting training...

📦 Downloading EMNIST dataset...


100%|██████████| 562M/562M [00:02<00:00, 230MB/s]


✅ Loaded 697932 training and 116323 test samples
   Classes: 47

🧠 Training deep neural network...
   Architecture: 784 → 512 → 256 → 128 → output
Iteration 1, loss = 0.52902161
Validation score: 0.868714
Iteration 2, loss = 0.35384883
Validation score: 0.876995
Iteration 3, loss = 0.31987391
Validation score: 0.882511
Iteration 4, loss = 0.30032365
Validation score: 0.884231
Iteration 5, loss = 0.28494408
Validation score: 0.887268
Iteration 6, loss = 0.27327706
Validation score: 0.885979
Iteration 7, loss = 0.26327234
Validation score: 0.886237
Iteration 8, loss = 0.25535472
Validation score: 0.889776
Iteration 9, loss = 0.24839294
Validation score: 0.888543
Iteration 10, loss = 0.24176162
Validation score: 0.888730
Iteration 11, loss = 0.23612800
Validation score: 0.888859
Iteration 12, loss = 0.23241240
Validation score: 0.888142
Iteration 13, loss = 0.22785297
Validation score: 0.887942
Iteration 14, loss = 0.22301116
Validation score: 0.886337
Iteration 15, loss = 0.21916288
Vali

## 4️⃣ Interactive Drawing Canvas

Draw a character and click **Recognize**!

In [ ]:
from google.colab import output
from IPython.display import display, HTML, Javascript
import base64
from io import BytesIO

# Ensure model is loaded
if 'trained_model' not in dir() or trained_model is None:
    trained_model = CharacterRecognitionModel()
    trained_model.load_model()

canvas_html = """
<style>
#canvas-container {
    display: flex; flex-direction: column; align-items: center; gap: 15px;
    padding: 20px; background: linear-gradient(135deg, #1a1a2e, #16213e);
    border-radius: 15px; max-width: 350px; margin: 20px auto;
}
#draw-canvas { border: 3px solid #4a90d9; border-radius: 10px; cursor: crosshair; }
.btn-container { display: flex; gap: 10px; }
.btn { padding: 12px 25px; border: none; border-radius: 8px; font-size: 16px;
       font-weight: bold; cursor: pointer; transition: transform 0.2s; }
.btn:hover { transform: translateY(-2px); }
.btn-clear { background: linear-gradient(135deg, #e74c3c, #c0392b); color: white; }
.btn-recognize { background: linear-gradient(135deg, #2ecc71, #27ae60); color: white; }
#result { font-size: 28px; font-weight: bold; color: #4a90d9; text-align: center; min-height: 40px; }
</style>

<div id="canvas-container">
    <h3 style="color: white; margin: 0;">✏️ Draw a Character</h3>
    <canvas id="draw-canvas" width="280" height="280"></canvas>
    <div class="btn-container">
        <button class="btn btn-clear" onclick="clearCanvas()">Clear</button>
        <button class="btn btn-recognize" onclick="recognize()">Recognize</button>
    </div>
    <div id="result">Draw something above!</div>
</div>

<script>
var canvas = document.getElementById('draw-canvas');
var ctx = canvas.getContext('2d');
var drawing = false, lastX, lastY;

ctx.fillStyle = 'black';
ctx.fillRect(0, 0, canvas.width, canvas.height);

function draw(x, y) {
    if (!drawing) return;
    ctx.beginPath();
    ctx.moveTo(lastX, lastY);
    ctx.lineTo(x, y);
    ctx.strokeStyle = 'white';
    ctx.lineWidth = 18;
    ctx.lineCap = 'round';
    ctx.stroke();
    lastX = x; lastY = y;
}

canvas.addEventListener('mousedown', e => { drawing = true; var r = canvas.getBoundingClientRect(); lastX = e.clientX - r.left; lastY = e.clientY - r.top; });
canvas.addEventListener('mousemove', e => { var r = canvas.getBoundingClientRect(); draw(e.clientX - r.left, e.clientY - r.top); });
canvas.addEventListener('mouseup', () => drawing = false);
canvas.addEventListener('mouseout', () => drawing = false);

canvas.addEventListener('touchstart', e => { e.preventDefault(); drawing = true; var r = canvas.getBoundingClientRect(); var t = e.touches[0]; lastX = t.clientX - r.left; lastY = t.clientY - r.top; });
canvas.addEventListener('touchmove', e => { e.preventDefault(); var r = canvas.getBoundingClientRect(); var t = e.touches[0]; draw(t.clientX - r.left, t.clientY - r.top); });
canvas.addEventListener('touchend', () => drawing = false);

function clearCanvas() { ctx.fillStyle = 'black'; ctx.fillRect(0, 0, canvas.width, canvas.height); document.getElementById('result').innerHTML = 'Draw something above!'; }
function recognize() { google.colab.kernel.invokeFunction('notebook.recognize', [canvas.toDataURL('image/png')], {}); }
</script>
"""

def recognize_callback(data_url):
    img_bytes = base64.b64decode(data_url.split(',')[1])
    img = Image.open(BytesIO(img_bytes)).convert('L')

    # Use EMNIST-specific preprocessing (with transpose!)
    processed = preprocess_image_for_emnist(img)
    prediction, _ = trained_model.predict(processed)

    display(Javascript(f'document.getElementById("result").innerHTML = "Prediction: <span style=\'font-size:48px;color:#2ecc71\'>{prediction}</span>";'))

output.register_callback('notebook.recognize', recognize_callback)
display(HTML(canvas_html))

## 5️⃣ Test with Uploaded Image

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

print("📤 Upload an image of a handwritten character:")
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(BytesIO(uploaded[filename]))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Original
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')

    # Preprocessed (before transpose)
    if img.mode != 'L':
        img_gray = img.convert('L')
    else:
        img_gray = img
    if np.mean(np.array(img_gray)) > 127:
        img_gray = ImageOps.invert(img_gray)
    axes[1].imshow(img_gray, cmap='gray')
    axes[1].set_title('Inverted')
    axes[1].axis('off')

    # Final preprocessed (transposed)
    processed = preprocess_image_for_emnist(img)
    axes[2].imshow(processed.reshape(28, 28), cmap='gray')
    axes[2].set_title('Preprocessed (28x28, Transposed)')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    prediction, _ = trained_model.predict(processed)
    print(f"\n🎯 Prediction: {prediction}")

## 6️⃣ Visualize on Test Samples

In [ ]:
import matplotlib.pyplot as plt

# Load test samples
test_dataset = EMNIST(root='./data', split='bymerge', train=False, download=True)
X_test = test_dataset.data.numpy()
y_test = test_dataset.targets.numpy()

# Random samples
np.random.seed(123)
indices = np.random.choice(len(X_test), 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()

correct = 0
for i, (idx, ax) in enumerate(zip(indices, axes)):
    img = X_test[idx]
    true_label = int(y_test[idx])

    # Note: Display transposed for human viewing
    ax.imshow(img.T, cmap='gray')

    # Predict (no transpose needed - model trained on transposed data)
    X_flat = img.reshape(1, 784).astype('float32') / 255.0
    pred, _ = trained_model.predict(X_flat)

    true_char = trained_model.mapping.get(true_label, str(true_label))
    is_correct = (pred == true_char)
    correct += is_correct

    color = 'green' if is_correct else 'red'
    ax.set_title(f'True: {true_char} | Pred: {pred}', color=color, fontsize=11)
    ax.axis('off')

plt.suptitle(f'Model Predictions ({correct}/12 correct)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7️⃣ Download Trained Model

In [ ]:
from google.colab import files

if os.path.exists('model.pkl'):
    print("📥 Downloading model.pkl...")
    files.download('model.pkl')
    print("✅ Place this file in your project folder to use with the desktop app.")
else:
    print("❌ No model file found. Run training first.")